# Tiny Code Agent for a Python Bug

Companion notebook for the [Building a Code Agent: Case Study](https://codemia.io/courses/applied_ai_systems/building_a_code_agent_case_study) lesson on Codemia.

**What you will build**: a code agent in about 200 lines of standard Python. It gets three tools,
`read_file`, `write_file` and `run_tests`, a small module with five planted bugs, and a loop that
keeps going until the test suite is green or the step budget runs out. You will read one full
iteration trace (test failure, read, patch, re-run) and then let the agent clear all five bugs.

## Setup

The notebook runs in two modes.

With an `OPENAI_API_KEY` set, a function-calling model decides which tool to call and writes the
patches itself. Without one, a scripted stand-in plays the model: it reads the same test output and
applies a known fix for whichever function failed. The stand-in is not smart, and that is the point.
Everything except the model call (tools, sandbox, loop, trace, stop conditions) is identical in
both modes, and that shared part is what this lesson is about.

In Colab, add your key under the key icon (Secrets) as `OPENAI_API_KEY`; the next cell picks it up.

In [ ]:
%pip install -q openai

In [ ]:
import json
import os
import re
import subprocess
import sys
import tempfile
import textwrap
from pathlib import Path

try:  # Colab keeps keys in Secrets rather than the environment
    from google.colab import userdata
    os.environ.setdefault('OPENAI_API_KEY', userdata.get('OPENAI_API_KEY') or '')
except Exception:
    pass

USE_REAL_LLM = bool(os.environ.get('OPENAI_API_KEY'))
MODEL = 'gpt-4o-mini'
print('mode:', 'live model' if USE_REAL_LLM else 'scripted stand-in')

## Concept recap

A code agent is a ReAct loop whose observations come from a real environment instead of a search
index. The environment is a working directory and a test command. Two properties make it work:

- **The tests are the reward signal.** The agent never decides it is done; the test runner does.
  A model that says "fixed!" while three tests still fail gets told so on the next step.
- **Writes are confined.** The agent can only touch files inside one throwaway directory, and the
  test command runs with a timeout. Model-written code is untrusted code.

## The buggy project

`stats_utils.py` holds six small functions and five planted bugs: an off-by-one denominator, an
even-length median that ignores half the list, a clamp that clamps to the wrong bound, a chunker
that drops its last partial chunk, and a word counter that splits on a single space. The tests are
written the way you would write them for real code, one behaviour per test.

In [ ]:
BUGGY_SOURCE = textwrap.dedent("""
    def mean(xs):
        return sum(xs) / (len(xs) - 1)


    def median(xs):
        s = sorted(xs)
        mid = len(s) // 2
        return s[mid]


    def clamp(x, lo, hi):
        return max(lo, min(x, lo))


    def chunk(xs, size):
        return [xs[i:i + size] for i in range(0, len(xs) - size, size)]


    def word_count(text):
        return len(text.split(' '))


    def is_palindrome(text):
        t = [c.lower() for c in text if c.isalnum()]
        return t == t[::-1]
""").lstrip()

In [ ]:
TEST_SOURCE = textwrap.dedent("""
    import unittest
    from stats_utils import mean, median, clamp, chunk, word_count, is_palindrome


    class TestStats(unittest.TestCase):
        def test_mean(self):
            self.assertEqual(mean([2, 4, 6]), 4)

        def test_median_odd(self):
            self.assertEqual(median([3, 1, 2]), 2)

        def test_median_even(self):
            self.assertEqual(median([4, 1, 3, 2]), 2.5)

        def test_clamp(self):
            self.assertEqual(clamp(15, 0, 10), 10)
            self.assertEqual(clamp(-3, 0, 10), 0)
            self.assertEqual(clamp(7, 0, 10), 7)

        def test_chunk(self):
            self.assertEqual(chunk([1, 2, 3, 4, 5], 2), [[1, 2], [3, 4], [5]])

        def test_word_count(self):
            self.assertEqual(word_count('  the quick\\tbrown  fox '), 4)
            self.assertEqual(word_count(''), 0)

        def test_palindrome(self):
            self.assertTrue(is_palindrome('A man, a plan, a canal: Panama'))
            self.assertFalse(is_palindrome('agent'))


    if __name__ == '__main__':
        unittest.main()
""").lstrip()

## Tools

Each tool is a plain function plus a JSON schema. The sandbox is a fresh temp directory, and
`_safe_path` refuses any path that resolves outside it, so `../../etc/passwd` or an absolute path
gets an error string back instead of a file. Errors are returned as text, not raised: the model
can read an error message and correct itself, but it cannot read a Python traceback that killed
the loop.

In [ ]:
def new_workspace():
    ws = Path(tempfile.mkdtemp(prefix='code_agent_'))
    (ws / 'stats_utils.py').write_text(BUGGY_SOURCE)
    (ws / 'test_stats_utils.py').write_text(TEST_SOURCE)
    return ws


def _safe_path(ws, path):
    p = (ws / path).resolve()
    if ws.resolve() not in p.parents:
        raise ValueError(f'path {path!r} is outside the workspace')
    return p


def read_file(ws, path):
    try:
        p = _safe_path(ws, path)
        lines = p.read_text().splitlines()
    except Exception as e:
        return f'ERROR: {e}'
    # Line numbers let the model point at a location without quoting the whole file back.
    return '\n'.join(f'{i:>3}| {line}' for i, line in enumerate(lines, 1))

In [ ]:
def write_file(ws, path, content):
    try:
        p = _safe_path(ws, path)
    except Exception as e:
        return f'ERROR: {e}'
    if p.name.startswith('test_'):
        # The cheapest way to make tests pass is to edit the tests. Forbid it in the tool,
        # not in the prompt, because a prompt is a request and a tool is a rule.
        return 'ERROR: test files are read-only'
    p.write_text(content)
    return f'wrote {len(content.splitlines())} lines to {path}'


def run_tests(ws):
    try:
        # -B: no .pyc cache. A one-word patch (lo -> hi) keeps the file size and lands in the
        # same second, so Python's mtime+size check would happily run the stale bytecode and
        # report the bug as still there. Agents hit this; humans rarely edit that fast.
        proc = subprocess.run([sys.executable, '-B', '-m', 'unittest', '-q'], cwd=ws,
                              capture_output=True, text=True, timeout=30)
    except subprocess.TimeoutExpired:
        return 'ERROR: tests timed out after 30s (infinite loop?)'
    out = (proc.stdout + proc.stderr).strip()
    # Long outputs eat the context window. Keep the head (which test failed) and the tail
    # (the summary line); the middle is usually repeated tracebacks.
    if len(out) > 2500:
        out = out[:1800] + '\n...[truncated]...\n' + out[-600:]
    return ('PASSED\n' if proc.returncode == 0 else 'FAILED\n') + out

In [ ]:
SCHEMAS = [
    {'type': 'function', 'function': {
        'name': 'read_file',
        'description': 'Read a file from the project. Returns it with line numbers.',
        'parameters': {'type': 'object', 'properties': {
            'path': {'type': 'string', 'description': 'Path relative to the project root'}},
            'required': ['path']}}},
    {'type': 'function', 'function': {
        'name': 'write_file',
        'description': 'Replace the whole content of a file. Send the complete file, '
                       'without line numbers. Test files cannot be written.',
        'parameters': {'type': 'object', 'properties': {
            'path': {'type': 'string'}, 'content': {'type': 'string'}},
            'required': ['path', 'content']}}},
    {'type': 'function', 'function': {
        'name': 'run_tests',
        'description': 'Run the unit tests. Returns PASSED or FAILED plus the output.',
        'parameters': {'type': 'object', 'properties': {}}}},
]


def dispatch(ws, name, args):
    if name == 'read_file':
        return read_file(ws, args.get('path', ''))
    if name == 'write_file':
        return write_file(ws, args.get('path', ''), args.get('content', ''))
    if name == 'run_tests':
        return run_tests(ws)
    return f'ERROR: unknown tool {name}'

Before giving these to a model, check them by hand. The test run below is the starting state the
agent will see: five failing tests out of seven, one per planted bug.

In [ ]:
ws = new_workspace()
print(read_file(ws, 'stats_utils.py').splitlines()[0:3])
print(write_file(ws, '../escape.py', 'x'))
print(write_file(ws, 'test_stats_utils.py', 'x'))
print(run_tests(ws)[-400:])

## The model call

One function hides which backend is answering and returns `(tool_calls, text)`.

The scripted stand-in follows the policy a careful engineer would: run the tests, read the file
the first time it needs to, then patch the function named in the first failing test and run the
tests again. It knows the correct fix for each function (that is the part a real model supplies),
and it only ever learns which function is broken from the test output.

In [ ]:
SYSTEM = (
    'You are a code agent fixing bugs in a small Python project. The project has '
    'stats_utils.py and test_stats_utils.py. Run the tests first. Fix one failing test at a '
    'time: read the code, write the corrected file, run the tests again. Do not edit tests. '
    'When the tests pass, reply with a one-line summary and no tool call.'
)

FIXES = {  # what the stand-in "knows": the corrected body for each function
    'mean': ('return sum(xs) / (len(xs) - 1)', 'return sum(xs) / len(xs)'),
    'median': ('    return s[mid]\n',
               '    if len(s) % 2:\n        return s[mid]\n'
               '    return (s[mid - 1] + s[mid]) / 2\n'),
    'clamp': ('min(x, lo)', 'min(x, hi)'),
    'chunk': ('len(xs) - size, size', 'len(xs), size'),
    'word_count': ("text.split(' ')", 'text.split()'),
}

In [ ]:
def _first_failing_function(test_output):
    m = re.search(r'(?:FAIL|ERROR): test_(\w+)', test_output)
    if not m:
        return None
    name = m.group(1)
    return next((f for f in FIXES if name.startswith(f)), None)


def call_scripted(messages):
    tool_msgs = [m for m in messages if m['role'] == 'tool']
    last = tool_msgs[-1] if tool_msgs else None
    if last is None or last['name'] == 'write_file':
        return [('run_tests', {})], 'Checking the current state of the tests.'
    if last['name'] == 'run_tests':
        if last['content'].startswith('PASSED'):
            return [], 'All tests pass.'
        fn = _first_failing_function(last['content'])
        return [('read_file', {'path': 'stats_utils.py'})], f'test for {fn}() fails; reading it.'
    if last['name'] == 'read_file':
        source = re.sub(r'^\s*\d+\| ?', '', last['content'], flags=re.M) + '\n'
        runs = [m for m in tool_msgs if m['name'] == 'run_tests']
        fn = _first_failing_function(runs[-1]['content'])
        old, new = FIXES[fn]
        return ([('write_file', {'path': 'stats_utils.py', 'content': source.replace(old, new, 1)})],
                f'patching {fn}().')
    return [], 'Stuck.'

In [ ]:
def call_openai(messages):
    from openai import OpenAI
    resp = OpenAI().chat.completions.create(model=MODEL, messages=messages, tools=SCHEMAS,
                                            temperature=0)
    msg = resp.choices[0].message
    calls = [(c.function.name, json.loads(c.function.arguments or '{}'), c.id)
             for c in (msg.tool_calls or [])]
    return calls, msg.content or ''


def call_llm(messages):
    if USE_REAL_LLM:
        return call_openai(messages)
    calls, text = call_scripted(messages)
    return [(n, a, f'call_{len(messages)}_{i}') for i, (n, a) in enumerate(calls)], text

## The loop

Per step: ask the model, run every tool it asked for, append each result as a `tool` message.
Stop when the model answers without a tool call **and** the tests last passed, or when the step
budget runs out. Checking the tests ourselves on the way out matters: a model can stop early and
say it is done, and the trace should record that it was wrong.

In [ ]:
def run_agent(ws, max_steps=30, verbose=True):
    messages = [{'role': 'system', 'content': SYSTEM},
                {'role': 'user', 'content': 'Make the test suite pass.'}]
    trace = []
    for step in range(1, max_steps + 1):
        calls, text = call_llm(messages)
        if text:
            trace.append((step, 'think', text))
        if not calls:
            break
        messages.append({'role': 'assistant', 'content': text or None, 'tool_calls': [
            {'id': cid, 'type': 'function',
             'function': {'name': n, 'arguments': json.dumps(a)}} for n, a, cid in calls]})
        for name, args, cid in calls:
            result = dispatch(ws, name, args)
            trace.append((step, name, result))
            messages.append({'role': 'tool', 'tool_call_id': cid, 'name': name, 'content': result})
    final = run_tests(ws)
    return {'steps': step, 'passed': final.startswith('PASSED'), 'trace': trace, 'final': final}

In [ ]:
def show(trace, first=None, width=96):
    for step, kind, body in trace[:first]:
        head = body.strip().splitlines()
        if kind == 'read_file':
            head = head[:4] + ['...']
        elif kind == 'run_tests':
            head = [head[0]] + [l for l in head if l.startswith(('FAIL:', 'ERROR:', 'Ran', 'OK'))]
        print(f'[{step:>2}] {kind.upper():<10} ' + head[0][:width])
        for line in head[1:8]:
            print(' ' * 16 + line[:width])

## Experiment 1: one full iteration

Give the agent a budget of four steps, enough for exactly one cycle: run tests, read, patch,
re-run. Read the trace line by line. With the stand-in, the second test run shows one fewer
failure than the first, and the diff shows the one line it changed. A live model often patches
every bug it can see in that single write, even though the system prompt asks for one at a time:
when we ran gpt-4o-mini here, one write took the suite from five failures to zero.

In [ ]:
import difflib

ws = new_workspace()
r = run_agent(ws, max_steps=4)
show(r['trace'])
print()
diff = difflib.unified_diff(BUGGY_SOURCE.splitlines(), (ws / 'stats_utils.py').read_text().splitlines(),
                            'before', 'after', lineterm='', n=1)
print('\n'.join(diff))

## Experiment 2: all five bugs

Same agent, fresh workspace, a budget of 30 steps. Count how many tool calls it took per bug.
The scripted stand-in needs exactly three per bug (read, write, re-run) plus the opening test run,
16 in all. A live model is usually cheaper on a project this small because it batches fixes, and
occasionally more expensive because a batched patch breaks something it did not mean to touch.

In [ ]:
ws = new_workspace()
r = run_agent(ws, max_steps=30, verbose=False)
calls = [k for _, k, _ in r['trace'] if k != 'think']
print('passed:', r['passed'], '| model calls:', r['steps'], '| tool calls:', len(calls))
for tool in ('run_tests', 'read_file', 'write_file'):
    print(f'  {tool:<11} {calls.count(tool)}')
print(r['final'].splitlines()[-1])

In [ ]:
# Failures remaining after each test run: the agent's learning curve, measured by the tests.
curve = []
for _, kind, body in r['trace']:
    if kind == 'run_tests':
        m = re.search(r'failures=(\d+)', body)
        e = re.search(r'errors=(\d+)', body)
        curve.append((int(m.group(1)) if m else 0) + (int(e.group(1)) if e else 0))
print('failing tests after each run:', curve)

## Variations

Three ways to break the agent, each removing one safeguard.

In [ ]:
# 1. Too small a budget. The stand-in spends 3 steps per bug plus the opening run, so 10 steps
# clears 3 bugs. A live model that batches its fixes can get through on 4.
for budget in (4, 7, 10, 16):
    ws = new_workspace()
    r = run_agent(ws, max_steps=budget, verbose=False)
    left = re.search(r'failures=(\d+)', r['final'])
    print(f'budget {budget:>2}: passed={r["passed"]!s:<5} failing={left.group(1) if left else 0}')

In [ ]:
# 2. What the read-only rule prevents. Make the edit an agent with a writable test file is
# tempted to make: change the expected value instead of the code. Green suite, same bug.
ws = new_workspace()
(ws / 'test_stats_utils.py').write_text(TEST_SOURCE.replace(
    'self.assertEqual(mean([2, 4, 6]), 4)', 'self.assertEqual(mean([2, 4, 6]), 6)'))
print('after editing the TEST, mean test is:',
      'green' if 'test_mean' not in run_tests(ws) else 'red')
print('mean([2, 4, 6]) still returns', eval('mean([2, 4, 6])',
      {}, {'mean': lambda xs: sum(xs) / (len(xs) - 1)}))

In [ ]:
# 3. No output truncation. One noisy test can flood the context window. Print how big a raw
# failing run is compared to what the model actually sees.
ws = new_workspace()
raw = subprocess.run([sys.executable, '-B', '-m', 'unittest', '-v'], cwd=ws,
                     capture_output=True, text=True).stderr
print('raw verbose output:', len(raw), 'chars | what run_tests returns:', len(run_tests(ws)), 'chars')

Things worth noticing:

1. **Ground truth comes from outside the model.** The loop ends because `run_tests` says PASSED,
   not because the model says so. In live mode, try deleting the final `run_tests` check in
   `run_agent` and see how often "fixed" is reported with tests still red.
2. **Most safety is in the tools.** The path check, the read-only test files and the timeout are
   each about three lines, and each one closes a failure you would otherwise have to prompt
   against and hope.
3. **"One at a time" is a request, not a rule.** The stand-in fixes the first failure and
   re-runs; a live model told to do the same often fixes everything in one write. That is cheaper
   when it works and harder to debug when it does not. If you need small steps, enforce them in
   the tool (reject a write that changes more than one function), the same way the test files
   are protected.

## What to try next

- Plant a bug whose fix breaks another test (for example, make `median` call `mean`), and watch
  how the loop handles a patch that turns one test green and another red.
- Replace `write_file` with an `edit_file(path, old, new)` tool. Whole-file writes are simple but
  cost tokens and invite accidental deletions; search-and-replace edits are what production
  agents use.
- Run the live agent five times on fresh workspaces and record tool calls per run. The spread is
  the real cost estimate for an agent, not the best run.